Bradley Belizaire
MARKETING BI & DECISION INTELLIGENCE PIPELINE ANALYTIQUE COMPLET 
Bachelor 3 YNOV — Advanced Marketing Analytics

IMPORTATION LIBRAIRIE 

In [ ]:
# ── Librairies standard 
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.options.display.max_columns = 50
pd.options.display.float_format = '{:.2f}'.format

# ── Visualisation 
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Palette cohérente
PALETTE = ['#e94560', '#0f3460', '#533483', '#05c46b', '#ffa41b']
sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams.update({'figure.dpi': 110, 'font.family': 'DejaVu Sans'})

# ── Machine Learning 
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, RocCurveDisplay,
                             ConfusionMatrixDisplay)

# ── Statistiques
from statsmodels.multivariate.manova import MANOVA
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy.stats import pearsonr, f_oneway
from statsmodels.stats.outliers_influence import variance_inflation_factor



CHARGEMENT ET AUDIT 

In [ ]:
df_raw = pd.read_csv('marketing_campaign.csv', sep=';', encoding='utf-8-sig')

print(f'Shape initial : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes')
df_raw.head(3)

In [ ]:
print('TYPOLOGIE DES VARIABLES')
type_map = {
    'Identifiant':      ['ID'],
    'Socio-démographie':['Year_Birth','Education','Marital_Status','Income','Kidhome','Teenhome'],
    'Comportement CRM': ['Dt_Customer','Recency','Complain'],
    'Dépenses produits':['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds'],
    'Canaux achat':     ['NumDealsPurchases','NumWebPurchases','NumCatalogPurchases','NumStorePurchases','NumWebVisitsMonth'],
    'Campagnes':        ['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3','AcceptedCmp4','AcceptedCmp5'],
    'Constantes (inutiles)':['Z_CostContact','Z_Revenue'],
    'Cible (target)':   ['Response']
}
for cat, cols in type_map.items():
    print(f'  {cat:30s} → {cols}')


valeur manquante 

In [ ]:

missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
df_missing = pd.DataFrame({'Manquants': missing, 'Pourcentage': missing_pct})
df_missing = df_missing[df_missing['Manquants'] > 0]

print('VALEURS MANQUANTES')
if df_missing.empty:
    print('  Aucune valeur manquante détectée... sauf :')
print(df_missing)
print()
print(f'  Income : {missing["Income"]} valeurs manquantes ({missing_pct["Income"]}% du dataset)')
print('   Stratégie : imputation par la médiane (robuste aux outliers)')


doublon

In [ ]:
 
n_dup = df_raw.duplicated().sum()
print(f'DOUBLONS')
print(f'  Lignes dupliquées : {n_dup}')
if n_dup == 0:
        print('Aucun doublon — dataset propre sur ce point.')

Anomalies de codage & valeurs aberrantes

In [ ]:

print('ANOMALIES DÉTECTÉES')

# Year_Birth
print(f'\n[1] Year_Birth — plage : {df_raw["Year_Birth"].min()} à {df_raw["Year_Birth"].max()}')
old = df_raw[df_raw['Year_Birth'] < 1930]
print(f'    Clients nés avant 1930 (âge > 94 ans) : {len(old)} lignes → aberrant, à supprimer')
print(df_raw[df_raw['Year_Birth'] < 1930][['ID','Year_Birth','Income','Education']].to_string())

# Income
print(f'\n[2] Income — max : {df_raw["Income"].max():,.0f}')
income_outliers = df_raw[df_raw['Income'] > 200000]
print(f'    Revenus > 200 000 : {len(income_outliers)} ligne(s) — outlier extrême, à supprimer')

# Marital_Status
print(f'\n[3] Marital_Status — modalités :')
print(df_raw['Marital_Status'].value_counts().to_string())
print("    → 'Alone', 'Absurd', 'YOLO' : valeurs incohérentes, à regrouper dans 'Single'")

# Colonnes constantes
print(f'\n[4] Colonnes constantes (zero variance) :')
for col in ['Z_CostContact','Z_Revenue']:
    print(f'    {col} : {df_raw[col].nunique()} valeur unique → colonne inutile, à supprimer')

# Campagnes — vérification des valeurs (doivent être 0 ou 1)
cmp_cols = ['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3','AcceptedCmp4','AcceptedCmp5','Response']
print(f'\n[5] Variables binaires campagnes — vérification :')
for col in cmp_cols:
    uniq = sorted(df_raw[col].unique())
    status = '✅' if uniq == [0,1] else '⚠️'
    print(f'    {status} {col} : {uniq}')

Visualisation de l'audit qualité

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Audit Qualité — Distribution des variables clés avant nettoyage', fontsize=14, fontweight='bold')

# Income brut
axes[0].hist(df_raw['Income'].dropna(), bins=50, color=PALETTE[0], edgecolor='white', linewidth=0.5)
axes[0].axvline(200000, color='black', linestyle='--', label='Seuil outlier (200k)')
axes[0].set_title('Distribution Income (brut)')
axes[0].set_xlabel('Income (€)')
axes[0].legend()

# Year_Birth
axes[1].hist(df_raw['Year_Birth'], bins=40, color=PALETTE[1], edgecolor='white', linewidth=0.5)
axes[1].axvline(1930, color='red', linestyle='--', label='Seuil (1930)')
axes[1].set_title('Distribution Year_Birth (brut)')
axes[1].set_xlabel('Année de naissance')
axes[1].legend()

# Marital Status
ms_counts = df_raw['Marital_Status'].value_counts()
colors = [PALETTE[3] if v >= 50 else PALETTE[0] for v in ms_counts.values]
axes[2].barh(ms_counts.index, ms_counts.values, color=colors)
axes[2].set_title('Marital Status — modalités')
axes[2].set_xlabel('Nombre de clients')

plt.tight_layout()
plt.savefig('01_audit_qualite.png', dpi=120, bbox_inches='tight')
plt.show()
print('📊 Graphique sauvegardé : 01_audit_qualite.png')

Nettoyage et feature engineering 

In [ ]:
df = df_raw.copy()

n_initial = len(df)

# [A] Imputation Income manquant par la médiane
income_median = df['Income'].median()
df['Income'] = df['Income'].fillna(income_median)
print(f'[A] Income : {df_raw["Income"].isnull().sum()} valeurs manquantes → imputées par la médiane ({income_median:,.0f}€)')

# [B] Suppression outlier Income > 200 000
df = df[df['Income'] < 200_000].copy()
print(f'[B] Income outliers (> 200k) : {n_initial - len(df)} ligne(s) supprimée(s)')

# [C] Suppression Year_Birth < 1930
n_before = len(df)
df = df[df['Year_Birth'] >= 1930].copy()
print(f'[C] Year_Birth < 1930 : {n_before - len(df)} ligne(s) supprimée(s)')

# [D] Nettoyage Marital_Status
df['Marital_Status'] = df['Marital_Status'].replace({'Alone': 'Single', 'Absurd': 'Single', 'YOLO': 'Single'})
print(f'[D] Marital_Status : modalités incohérentes regroupées dans "Single"')

# [E] Suppression colonnes constantes
df = df.drop(columns=['Z_CostContact', 'Z_Revenue'])
print(f'[E] Colonnes Z_CostContact et Z_Revenue supprimées (variance nulle)')

print(f'\n✅ Dataset nettoyé : {len(df)} lignes × {len(df.columns)} colonnes (perte de {n_initial - len(df)} lignes, soit {(n_initial-len(df))/n_initial*100:.1f}%)')

Feature Engineering 

In [ ]:
# Date de référence = date d'inscription max dans le dataset
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'])
REF_DATE = df['Dt_Customer'].max()

# --- Variables construites ---
df['Age']             = 2024 - df['Year_Birth']                               # Âge du client
df['CustomerDays']    = (REF_DATE - df['Dt_Customer']).dt.days                # Ancienneté CRM
df['TotalSpend']      = df[['MntWines','MntFruits','MntMeatProducts',
                              'MntFishProducts','MntSweetProducts','MntGoldProds']].sum(axis=1)
df['TotalPurchases']  = df[['NumWebPurchases','NumCatalogPurchases','NumStorePurchases']].sum(axis=1)
df['TotalAccepted']   = df[['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3',
                              'AcceptedCmp4','AcceptedCmp5']].sum(axis=1)
df['TotalKids']       = df['Kidhome'] + df['Teenhome']                        # Nb enfants total
df['SpendPerIncome']  = df['TotalSpend'] / df['Income']                       # Effort d'achat relatif
df['HasChildren']     = (df['TotalKids'] > 0).astype(int)                     # Indicateur binaire
df['IsCouple']        = df['Marital_Status'].isin(['Married','Together']).astype(int)
df['EducLevel']       = df['Education'].map({'Basic':0,'2n Cycle':1,'Graduation':2,'Master':3,'PhD':4})

# Part de chaque canal d'achat
df['PctWebPurchases']     = df['NumWebPurchases']     / df['TotalPurchases'].replace(0,1)
df['PctCatalogPurchases'] = df['NumCatalogPurchases'] / df['TotalPurchases'].replace(0,1)
df['PctStorePurchases']   = df['NumStorePurchases']   / df['TotalPurchases'].replace(0,1)

print('=== VARIABLES CONSTRUITES ===')
new_vars = ['Age','CustomerDays','TotalSpend','TotalPurchases','TotalAccepted',
            'TotalKids','SpendPerIncome','HasChildren','IsCouple','EducLevel']
print(df[new_vars].describe().round(2).to_string())


Analyse Multidimensionnelle 

In [ ]:
#Statistiques descriptives par segment socio-démo
print('PROFIL MOYEN PAR NIVEAU D\'ÉDUCATION')
educ_profile = df.groupby('Education')[['Income','TotalSpend','Age','TotalAccepted','Response']].mean().round(1)
educ_profile.columns = ['Revenu Moy.','Dépense Moy.','Âge Moy.','Campagnes acc.','Taux Réponse']
educ_profile['Taux Réponse'] = (educ_profile['Taux Réponse']*100).round(1).astype(str) + '%'
print(educ_profile.to_string())

print('\n=== PROFIL MOYEN PAR STATUT MARITAL ===')
marital_profile = df.groupby('Marital_Status')[['Income','TotalSpend','TotalKids','Response']].mean().round(2)
print(marital_profile.to_string())

Matrice de corrélation

In [ ]:
num_cols = ['Age','Income','TotalSpend','TotalPurchases','Recency','TotalAccepted',
            'TotalKids','CustomerDays','SpendPerIncome','Response']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Masque triangle supérieur
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Matrice de Corrélation — Variables clés du dataset marketing', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('02_correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nVARIABLES LES PLUS CORRÉLÉES AVEC RESPONSE')
corr_response = corr['Response'].drop('Response').sort_values(key=abs, ascending=False)
for var, val in corr_response.items():
    direction = '↑' if val > 0 else '↓'
    print(f'  {direction} {var:25s} : r = {val:.3f}')

ANOVA: L'éducation influence-t-elle les dépenses ?

In [ ]:
print('ANOVA : Niveau d\'éducation → TotalSpend')
groups = [df[df['Education'] == edu]['TotalSpend'].values for edu in df['Education'].unique()]
f_stat, p_value = f_oneway(*groups)
print(f'  Statistique F : {f_stat:.4f}')
print(f'  p-value      : {p_value:.2e}')
if p_value < 0.05:
    print('  ✅ Résultat : Le niveau d\'éducation a un effet SIGNIFICATIF sur les dépenses (p < 0.05)')
    print('     → Les clients PhD et Master dépensent significativement plus.')
else:
    print('  ❌ Pas d\'effet significatif')

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

educ_order = ['Basic','2n Cycle','Graduation','Master','PhD']
df_plot = df[df['Education'].isin(educ_order)]

sns.boxplot(data=df_plot, x='Education', y='TotalSpend', order=educ_order,
            palette=PALETTE[:5], ax=axes[0])
axes[0].set_title(f'ANOVA : Éducation → Dépenses\n(F={f_stat:.1f}, p={p_value:.2e})', fontweight='bold')
axes[0].set_xlabel('Niveau d\'éducation')
axes[0].set_ylabel('Dépenses totales (€)')

# Income vs TotalSpend coloré par Response
for i, resp in enumerate([0, 1]):
    subset = df[df['Response'] == resp]
    axes[1].scatter(subset['Income'], subset['TotalSpend'],
                    c=PALETTE[i], alpha=0.4, s=15,
                    label='Non-répondant' if resp == 0 else 'Répondant')
axes[1].set_title('Income vs Dépenses — coloré par Réponse campagne', fontweight='bold')
axes[1].set_xlabel('Revenu (€)')
axes[1].set_ylabel('Dépenses totales (€)')
axes[1].legend()

plt.tight_layout()
plt.savefig('03_anova_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

MANOVA :L'éducation influence-t-elle plusieurs variables simultanément

In [ ]:
print('MANOVA : Éducation → (Income, TotalSpend, TotalPurchases)')
fit = MANOVA.from_formula('Income + TotalSpend + TotalPurchases ~ Education', data=df)
result = fit.mv_test()
print(result)
print('\n📌 Interprétation : Si p < 0.05 → le niveau d\'éducation a un effet global')
print('   significatif sur la combinaison Income + Dépenses + Nb achats.')


Corrélation partielle La relation entre le revenu et la réponse à une campagne             est-elle réelle, ou médiée par les dépenses 

In [ ]:
print('CORRÉLATION PARTIELLE : Income → Response (contrôle : TotalSpend)')

r_XY, _ = pearsonr(df['Income'], df['Response'])        # Income ↔ Response
r_XZ, _ = pearsonr(df['Income'], df['TotalSpend'])      # Income ↔ TotalSpend
r_YZ, _ = pearsonr(df['Response'], df['TotalSpend'])    # Response ↔ TotalSpend

# Formule de la corrélation partielle
r_partial = (r_XY - r_XZ * r_YZ) / (np.sqrt(1 - r_XZ**2) * np.sqrt(1 - r_YZ**2))

print(f'  Corrélation simple   Income ↔ Response   : r = {r_XY:.4f}')
print(f'  Corrélation partielle (contrôle TotalSpend) : r = {r_partial:.4f}')
print()
if abs(r_partial) < abs(r_XY):
    print('  📌 La corrélation partielle est plus faible → les dépenses médient partiellement')
    print('     la relation entre revenu et réponse à la campagne.')
    print('     → Ce n\'est pas le revenu seul qui explique la réponse, mais le comportement d\'achat.')


Contrôle de la multicolinéarité

In [ ]:
print(' VIF — Détection de multicolinéarité')
features_vif = ['Age','Income','TotalSpend','Recency','TotalAccepted','TotalKids','CustomerDays']
X_vif = df[features_vif].dropna()
X_vif_c = sm.add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif_c.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif_c.values, i) for i in range(X_vif_c.shape[1])]
vif_data = vif_data[vif_data['Variable'] != 'const'].sort_values('VIF', ascending=False)

print(vif_data.to_string(index=False))
print()
print('  Seuil VIF > 10 : multicolinéarité forte (problématique pour les modèles)')
high_vif = vif_data[vif_data['VIF'] > 10]
if not high_vif.empty:
    print(f'  ⚠️  Variables à VIF élevé : {high_vif["Variable"].tolist()}')
else:
    print('  ✅ Aucune variable à VIF > 10 — pas de multicolinéarité problématique.')


Analyse comportementale — Canaux d'achat & catégories produits

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Analyse comportementale des clients', fontsize=14, fontweight='bold')

# Part des canaux d'achat
channels = {'Web': df['NumWebPurchases'].mean(),
            'Catalogue': df['NumCatalogPurchases'].mean(),
            'Magasin': df['NumStorePurchases'].mean(),
            'Promotions': df['NumDealsPurchases'].mean()}
bars = axes[0].bar(channels.keys(), channels.values(), color=PALETTE[:4], edgecolor='white')
for bar, val in zip(bars, channels.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Nombre moyen d\'achats par canal')
axes[0].set_ylabel('Achats moyens')

# Dépenses par catégorie
categories = {'Vins': df['MntWines'].mean(), 'Viandes': df['MntMeatProducts'].mean(),
               'Poissons': df['MntFishProducts'].mean(), 'Or': df['MntGoldProds'].mean(),
               'Fruits': df['MntFruits'].mean(), 'Sucreries': df['MntSweetProducts'].mean()}
sorted_cats = dict(sorted(categories.items(), key=lambda x: x[1], reverse=True))
bars2 = axes[1].barh(list(sorted_cats.keys()), list(sorted_cats.values()),
                      color=PALETTE[:6], edgecolor='white')
for bar, val in zip(bars2, sorted_cats.values()):
    axes[1].text(val + 2, bar.get_y() + bar.get_height()/2,
                 f'{val:.0f}€', va='center', fontweight='bold')
axes[1].set_title('Dépense moyenne par catégorie produit')
axes[1].set_xlabel('Dépense moyenne (€, 2 ans)')

plt.tight_layout()
plt.savefig('04_comportement_achat.png', dpi=120, bbox_inches='tight')
plt.show()

print('📌 Le magasin est le canal privilégié. Les vins et viandes représentent ~75% des dépenses.')


Segmentation K-MEANS 

In [ ]:
CLUSTER_FEATURES = ['Income', 'TotalSpend', 'Recency', 'TotalAccepted', 'TotalKids', 'Age']

print('Variables utilisées pour la segmentation :')
for f in CLUSTER_FEATURES:
    print(f'  • {f}')
print('\nJustification : ces 6 variables capturent les 3 dimensions clés')
print('  - Pouvoir d\'achat     : Income, TotalSpend')
print('  - Engagement CRM      : Recency, TotalAccepted')
print('  - Profil socio-démo   : TotalKids, Age')

X_clust = df[CLUSTER_FEATURES].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clust)
print(f'\n✅ Normalisation StandardScaler appliquée sur {X_scaled.shape[0]} clients')


Méthode du coude (Elbow) + Silhouette Score

In [ ]:
from sklearn.metrics import silhouette_score

K_range = range(2, 9)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Détermination du nombre optimal de clusters', fontsize=13, fontweight='bold')

axes[0].plot(K_range, inertias, 'o-', color=PALETTE[0], linewidth=2, markersize=8)
axes[0].axvline(x=4, color='black', linestyle='--', alpha=0.6, label='k=4 choisi')
axes[0].set_title('Méthode du Coude (Elbow)')
axes[0].set_xlabel('Nombre de clusters (k)')
axes[0].set_ylabel('Inertie (WCSS)')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 's-', color=PALETTE[1], linewidth=2, markersize=8)
axes[1].axvline(x=4, color='black', linestyle='--', alpha=0.6, label='k=4 choisi')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Nombre de clusters (k)')
axes[1].set_ylabel('Score de silhouette')
axes[1].legend()

plt.tight_layout()
plt.savefig('05_elbow_silhouette.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Silhouette scores : { {k: round(s,3) for k,s in zip(K_range, silhouettes)} }')
print('→ k=4 retenu : bon compromis coude + interprétabilité métier')

Entraînement du modèle final K-Means 

In [ ]:
K_FINAL = 4
km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df['Segment'] = km_final.fit_predict(X_scaled)

# Profil de chaque segment
seg_profile = df.groupby('Segment').agg(
    N_clients       =('Segment','count'),
    Income_moy      =('Income','mean'),
    Depense_moy     =('TotalSpend','mean'),
    Recency_moy     =('Recency','mean'),
    Age_moy         =('Age','mean'),
    Kids_moy        =('TotalKids','mean'),
    Cmp_acceptees   =('TotalAccepted','mean'),
    Taux_reponse    =('Response','mean'),
).round(2)
seg_profile['Taux_reponse_%'] = (seg_profile['Taux_reponse']*100).round(1)
seg_profile['Part_%'] = (seg_profile['N_clients']/len(df)*100).round(1)
print('=== PROFIL DES SEGMENTS ===')
print(seg_profile.to_string())


Nommage métier des segments

In [ ]:
# Tri par Income moyen décroissant pour nommer de façon cohérente
seg_income_rank = seg_profile['Income_moy'].sort_values(ascending=False)
seg_spend_rank  = seg_profile['Depense_moy'].sort_values(ascending=False)

print('Classement des segments par revenu moyen :')
print(seg_income_rank)
print()
print('Classement des segments par dépenses moyennes :')
print(seg_spend_rank)

# Nommage automatique basé sur les rangs
seg_labels = {
    seg_spend_rank.index[0]: 'Champions',      # High spend, high income
    seg_spend_rank.index[1]: 'Fidèles Actifs',  # Bon spend, engagés
    seg_spend_rank.index[2]: 'Clients Modérés', # Income/spend moyen
    seg_spend_rank.index[3]: 'Petits Budgets',  # Faible spend
}
df['SegmentLabel'] = df['Segment'].map(seg_labels)

print('\nMAPPING SEGMENTS → LABELS MÉTIER ')
for seg_id, label in seg_labels.items():
    row = seg_profile.loc[seg_id]
    print(f'  Segment {seg_id} → "{label}"')
    print(f'     Revenu moy : {row["Income_moy"]:,.0f}€ | Dépense moy : {row["Depense_moy"]:,.0f}€ | Taux réponse : {row["Taux_reponse_%"]}%')


Visualisation des segments 

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Visualisation de la Segmentation Clients', fontsize=14, fontweight='bold')

# Scatter PCA coloré par segment
for seg_id in sorted(df['Segment'].unique()):
    mask = df['Segment'] == seg_id
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=PALETTE[seg_id], s=15, alpha=0.5,
                    label=seg_labels[seg_id])

# Centroids
centroids_pca = pca.transform(km_final.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                c='black', marker='X', s=200, zorder=5, label='Centroids')
axes[0].set_title('Projection PCA 2D des 4 segments')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].legend(fontsize=9)

# Radar chart (Income, Dépenses, Réponse, Ancienneté)
radar_vars = ['Income_moy','Depense_moy','Taux_reponse_%','Recency_moy']
seg_radar = seg_profile[radar_vars].copy()
# Normalisation 0-100 pour le radar
for col in radar_vars:
    seg_radar[col] = (seg_radar[col] - seg_radar[col].min()) / (seg_radar[col].max() - seg_radar[col].min()) * 100

x_pos = np.arange(len(radar_vars))
width = 0.2
for i, (seg_id, label) in enumerate(seg_labels.items()):
    axes[1].bar(x_pos + i*width, seg_radar.loc[seg_id, radar_vars],
                width=width, label=label, color=PALETTE[seg_id], alpha=0.85)

axes[1].set_xticks(x_pos + width * 1.5)
axes[1].set_xticklabels(['Revenu', 'Dépenses', 'Taux réponse', 'Recency'], fontsize=10)
axes[1].set_title('Profil comparatif des segments (score 0-100)')
axes[1].set_ylabel('Score relatif (0-100)')
axes[1].legend(fontsize=9)


KPI METIER ET LECTURE DECISIONNEL 

In [ ]:
#KPI GLOBAUX 
print('=' * 55)
print('       KPI MARKETING GLOBAUX — VUE EXÉCUTIVE')
print('=' * 55)

kpi_global = {
    'Nb clients (après nettoyage)'        : len(df),
    'Taux de réponse global (%)'          : round(df['Response'].mean() * 100, 1),
    'Revenu moyen (€)'                    : round(df['Income'].mean(), 0),
    'Dépense totale moyenne (€, 2 ans)'   : round(df['TotalSpend'].mean(), 0),
    'Dépense mensuelle moyenne (€)'       : round(df['TotalSpend'].mean() / 24, 1),
    'Recency moyenne (jours)'             : round(df['Recency'].mean(), 1),
    'Ancienneté moyenne (jours)'          : round(df['CustomerDays'].mean(), 0),
    'Taux plainte global (%)'             : round(df['Complain'].mean() * 100, 2),
    'Taux acceptation au moins 1 cmp (%)'  : round((df['TotalAccepted'] > 0).mean() * 100, 1),
    'Nb moyen achats / client'            : round(df['TotalPurchases'].mean(), 1),
}
for kpi, val in kpi_global.items():
    print(f'  {kpi:45s} : {val}')

print('\n' + '=' * 55)
print('       KPI PAR CAMPAGNE')
print('=' * 55)
for i, cmp in enumerate(['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3','AcceptedCmp4','AcceptedCmp5']):
    rate = df[cmp].mean() * 100
    print(f'  Campagne {i+1} — Taux d\'acceptation : {rate:.1f}%')
print(f'  Campagne 6 (dernière) — Taux de réponse : {df["Response"].mean()*100:.1f}%')


KPI par segment

In [ ]:
print('KPI PAR SEGMENT MÉTIER')
seg_kpi = df.groupby('SegmentLabel').agg(
    N                    =('Segment','count'),
    Revenu_moy           =('Income','mean'),
    Depense_moy_2ans     =('TotalSpend','mean'),
    Depense_mensuelle    =('TotalSpend', lambda x: x.mean()/24),
    Recency_moy          =('Recency','mean'),
    Taux_reponse_pct     =('Response', lambda x: x.mean()*100),
    Cmp_acceptees_moy    =('TotalAccepted','mean'),
    Canal_fav            =('NumStorePurchases','mean'),
).round(1)

print(seg_kpi.to_string())


Score de valeur client

In [ ]:
# R = Recency (inversé : moins récent = score faible)
# F = Fréquence achats (TotalPurchases)
# M = Monetary (TotalSpend)

def rfm_score(series, n=4, ascending=True):
    """Découpage en quartiles → score 1 à 4."""
    labels = list(range(1, n+1)) if ascending else list(range(n, 0, -1))
    return pd.qcut(series, q=n, labels=labels, duplicates='drop')

df['R_score'] = rfm_score(df['Recency'],       ascending=False)  # Moins de jours = mieux
df['F_score'] = rfm_score(df['TotalPurchases'],ascending=True)
df['M_score'] = rfm_score(df['TotalSpend'],    ascending=True)

df['RFM_score'] = (df['R_score'].astype(float) +
                   df['F_score'].astype(float) +
                   df['M_score'].astype(float)) / 3 * 100 / 4  # Normalisé 0-100

print('SCORE RFM PAR SEGMENT')
rfm_by_seg = df.groupby('SegmentLabel')['RFM_score'].agg(['mean','min','max']).round(1)
rfm_by_seg.columns = ['Score moyen','Score min','Score max']
print(rfm_by_seg.to_string())

print(f'\nMoyenne globale RFM : {df["RFM_score"].mean():.1f}/100')
print('→ Score RFM utilisable pour prioriser les clients à cibler en campagne.')


SCORE PREDICTIF 

In [ ]:
FEATURES_MODEL = ['Age','Income','TotalSpend','Recency','TotalAccepted',
                   'TotalKids','CustomerDays','SpendPerIncome','IsCouple','EducLevel']
TARGET = 'Response'

X = df[FEATURES_MODEL]
y = df[TARGET]

# Split train/test stratifié (préserve la proportion 85/15)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisation
sc_model = StandardScaler()
X_train_s = sc_model.fit_transform(X_train)
X_test_s  = sc_model.transform(X_test)

print(f'Train : {len(X_train)} clients ({y_train.mean()*100:.1f}% répondants)')
print(f'Test  : {len(X_test)} clients ({y_test.mean()*100:.1f}% répondants)')
print(f'Features utilisées : {FEATURES_MODEL}')


Entraînement ET évaluation

In [ ]:
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, C=1.0)
lr_model.fit(X_train_s, y_train)

y_pred = lr_model.predict(X_test_s)
y_prob = lr_model.predict_proba(X_test_s)[:, 1]

auc = roc_auc_score(y_test, y_prob)
cv_scores = cross_val_score(lr_model, sc_model.transform(X), y, cv=5, scoring='roc_auc')

print('=== RÉSULTATS DU MODÈLE ===')
print(f'  AUC-ROC (test)          : {auc:.4f}')
print(f'  AUC-ROC (cross-val 5CV) : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print()
print('=== RAPPORT DE CLASSIFICATION ===')
print(classification_report(y_test, y_pred, target_names=['Non-répondant','Répondant']))
print('📌 Le recall sur les Répondants est prioritaire : on préfère cibler trop large que rater un répondant.')


Courbe ROC + Matrice de confusion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Évaluation du modèle de scoring — Régression Logistique', fontsize=13, fontweight='bold')

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0],
                                  color=PALETTE[0], name=f'Logistic Regression (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1], 'k--', alpha=0.5, label='Aléatoire')
axes[0].set_title('Courbe ROC')
axes[0].legend()

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Non-répondant','Répondant'],
    cmap='Blues', ax=axes[1]
)
axes[1].set_title('Matrice de Confusion')

plt.tight_layout()
plt.savefig('07_scoring_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()


Importance des features 

In [ ]:
coef_df = pd.DataFrame({
    'Variable' : FEATURES_MODEL,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=True)

colors = [PALETTE[0] if v < 0 else PALETTE[1] for v in coef_df['Coefficient']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(coef_df['Variable'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Importance des variables — Régression Logistique\n(coefficients standardisés)', fontweight='bold')
ax.set_xlabel('Coefficient (impact sur la probabilité de réponse)')

# Annotations
for bar, val in zip(bars, coef_df['Coefficient']):
    ax.text(val + (0.02 if val >= 0 else -0.02), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

legend_patches = [
    mpatches.Patch(color=PALETTE[1], label='↑ Augmente la probabilité de réponse'),
    mpatches.Patch(color=PALETTE[0], label='↓ Diminue la probabilité de réponse'),
]
ax.legend(handles=legend_patches, loc='lower right')
plt.tight_layout()
plt.savefig('08_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nINTERPRÉTATION MÉTIER DES COEFFICIENTS')
for _, row in coef_df.sort_values('Coefficient', key=abs, ascending=False).iterrows():
    direction = '✅ favorise' if row['Coefficient'] > 0 else '❌ réduit'
    print(f'  {row["Variable"]:20s} ({row["Coefficient"]:+.3f}) → {direction} la réponse')


 Application du score à tous les clients

In [ ]:
X_all_scaled = sc_model.transform(df[FEATURES_MODEL])
df['ScoreCampagne'] = lr_model.predict_proba(X_all_scaled)[:, 1]
df['ScoreCampagne_100'] = (df['ScoreCampagne'] * 100).round(1)

# Décile de score
df['DecileScore'] = pd.qcut(df['ScoreCampagne'], q=10, labels=False, duplicates='drop') + 1

print('DISTRIBUTION DU SCORE PAR SEGMENT')
score_by_seg = df.groupby('SegmentLabel')['ScoreCampagne_100'].agg(['mean','median','min','max']).round(1)
print(score_by_seg.to_string())

print('\n ANALYSE PAR DÉCILE (Top 30% = déciles 8-10)')
decile_analysis = df.groupby('DecileScore').agg(
    N=('Response','count'),
    Taux_reponse=('Response','mean'),
    Score_moy=('ScoreCampagne_100','mean')
).round(3)
decile_analysis['Taux_%'] = (decile_analysis['Taux_reponse']*100).round(1)
print(decile_analysis[['N','Score_moy','Taux_%']].to_string())
print('→ Cibler les déciles 8-10 maximise le ROI campagne.')


ROBUSTESSE ET LIMITES ANALYTIQUES

In [ ]:
print('TEST DE SENSIBILITÉ — Impact des outliers Income')

# Version avec outliers (seuil à 150k au lieu de 200k)
df_strict = df[df['Income'] < 150_000].copy()
X_strict = df_strict[FEATURES_MODEL]
y_strict = df_strict[TARGET]
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_strict, y_strict, test_size=0.2, random_state=42, stratify=y_strict)
sc2 = StandardScaler()
lr2 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr2.fit(sc2.fit_transform(X_tr2), y_tr2)
auc_strict = roc_auc_score(y_te2, lr2.predict_proba(sc2.transform(X_te2))[:,1])

print(f'  AUC original (seuil 200k) : {auc:.4f}')
print(f'  AUC strict   (seuil 150k) : {auc_strict:.4f}')
print(f'  Différence                : {abs(auc - auc_strict):.4f}')
print()
if abs(auc - auc_strict) < 0.01:
    print('✅ Résultat robuste : l\'AUC varie de moins de 1 point selon le seuil outlier retenu.')
else:
    print('⚠️  Sensibilité détectée — à mentionner dans les limites.')


Risque de data leakage 

In [53]:
print('RISQUE DE FUITE D\'INFORMATION (Data Leakage)')
print()
print('Variables écartées du modèle pour éviter le leakage :')
leakage_risk = {
    'AcceptedCmp1/2/3/4/5' : 'Ces variables sont les réponses aux campagnes précédentes,'
                               ' corrélées à Response mais disponibles AVANT la campagne → conservées car utiles',
    'TotalAccepted'         : 'Variable agrégée des campagnes passées → signal légitime de comportement',
    'Response'              : 'C\'est la cible → exclue des features par construction'
}
for var, comment in leakage_risk.items():
    print(f'  • {var:25s} : {comment}')

print()
print('=== LIMITES DU DATASET ===')
limites = [
    '1. Population unique (2240 clients) : résultats à valider sur un dataset plus large',
    '2. Snapshot statique : pas d\'évolution temporelle des comportements',
    '3. Déséquilibre classes (85/15) : géré par class_weight mais à surveiller',
    '4. Variables manquantes sur Income (1%) : imputation par médiane peut introduire un biais léger',
    '5. Pas d\'information géographique : segmentation aveugle au contexte local',
    '6. Campagnes numérotées 1-5 sans contexte temporel clair',
]
for l in limites:
    print(f'  {l}')


RISQUE DE FUITE D'INFORMATION (Data Leakage)

Variables écartées du modèle pour éviter le leakage :
  • AcceptedCmp1/2/3/4/5      : Ces variables sont les réponses aux campagnes précédentes, corrélées à Response mais disponibles AVANT la campagne → conservées car utiles
  • TotalAccepted             : Variable agrégée des campagnes passées → signal légitime de comportement
  • Response                  : C'est la cible → exclue des features par construction

=== LIMITES DU DATASET ===
  1. Population unique (2240 clients) : résultats à valider sur un dataset plus large
  2. Snapshot statique : pas d'évolution temporelle des comportements
  3. Déséquilibre classes (85/15) : géré par class_weight mais à surveiller
  4. Variables manquantes sur Income (1%) : imputation par médiane peut introduire un biais léger
  5. Pas d'information géographique : segmentation aveugle au contexte local
  6. Campagnes numérotées 1-5 sans contexte temporel clair


COMPLEMENT DASHBORD

In [ ]:
import pandas as pd
import numpy as np

# ── Chargement des données ────────────────────────────────────────
df = pd.read_csv("marketing_campaign.csv", sep=";", encoding="utf-8-sig")

# Nettoyage
df = df.dropna(subset=["Income"]).copy()

# Âge
df["Age"] = 2025 - df["Year_Birth"]

# Dépense totale
spend_cols = [
    "MntWines", "MntFruits", "MntMeatProducts",
    "MntFishProducts", "MntSweetProducts", "MntGoldProds"
]
df["TotalSpend"] = df[spend_cols].sum(axis=1)

# Fréquence d'achat
df["PurchaseFrequency"] = (
    df["NumWebPurchases"] +
    df["NumCatalogPurchases"] +
    df["NumStorePurchases"]
)

# Score campagne sur 100
campaign_cols = [
    "AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3",
    "AcceptedCmp4", "AcceptedCmp5", "Response"
]
df["ScoreCampagne_100"] = df[campaign_cols].sum(axis=1) / len(campaign_cols) * 100

# Score RFM simplifié
df["R_score"] = pd.qcut(
    df["Recency"].rank(method="first"),
    4,
    labels=[4, 3, 2, 1]
).astype(int)

df["F_score"] = pd.qcut(
    df["PurchaseFrequency"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
).astype(int)

df["M_score"] = pd.qcut(
    df["TotalSpend"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
).astype(int)

df["RFM_score"] = (df["R_score"] + df["F_score"] + df["M_score"]) / 12 * 100

# Segmentation simple
income_med = df["Income"].median()
spend_med = df["TotalSpend"].median()

conditions = [
    (df["Income"] >= income_med) & (df["TotalSpend"] >= spend_med),
    (df["Income"] >= income_med) & (df["TotalSpend"] < spend_med),
    (df["Income"] < income_med) & (df["TotalSpend"] >= spend_med),
]
choices = ["Premium", "Aisés prudents", "Bons acheteurs"]

df["SegmentLabel"] = np.select(conditions, choices, default="Standard")

PALETTE = ["#e94560", "#0f3460", "#533483", "#05c46b"]
SEGMENTS = sorted(df["SegmentLabel"].dropna().unique().tolist())

In [ ]:
DASHBORD 

In [ ]:
# dashboard.py — Marketing Analytics Dashboard
# Lancer avec : python dashboard.py
# Accéder sur : http://127.0.0.1:8050

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, dcc, html, Input, Output, dash_table
import dash_bootstrap_components as dbc

# ── Chargement des données enrichies ────────────────────────────────────────
df = pd.read_csv("marketing_clean_segmented.csv", sep=";", encoding="utf-8-sig")

PALETTE = ["#e94560", "#0f3460", "#533483", "#05c46b"]
SEGMENTS = sorted(df["SegmentLabel"].unique().tolist())

# ── App ──────────────────────────────────────────────────────────────────────
app = Dash(__name__, external_stylesheets=[dbc.themes.DARKLY])
server = app.server
app.title = "Marketing BI Dashboard"

# ── Layout ───────────────────────────────────────────────────────────────────
app.layout = dbc.Container([

    # Header
    dbc.Row([
        dbc.Col([
            html.H1("📊 Marketing Analytics Dashboard",
                    className="text-center my-3",
                    style={"color": "#e94560", "fontWeight": "bold"}),
            html.P("Business Intelligence & Decision Intelligence — Bradley Belizaire",
                   className="text-center text-muted mb-4")
        ])
    ]),

    # KPI Cards
    dbc.Row([
        dbc.Col(dbc.Card([
            dbc.CardBody([
                html.H4(f"{len(df):,}", className="card-title text-center", style={"color":"#05c46b","fontSize":"2em"}),
                html.P("Clients analysés", className="card-text text-center text-muted")
            ])
        ], color="dark", outline=True), width=3),

        dbc.Col(dbc.Card([
            dbc.CardBody([
                html.H4(f"{df['Response'].mean()*100:.1f}%", className="card-title text-center", style={"color":"#e94560","fontSize":"2em"}),
                html.P("Taux de réponse global", className="card-text text-center text-muted")
            ])
        ], color="dark", outline=True), width=3),

        dbc.Col(dbc.Card([
            dbc.CardBody([
                html.H4(f"{df['TotalSpend'].mean():.0f}€", className="card-title text-center", style={"color":"#ffa41b","fontSize":"2em"}),
                html.P("Dépense moyenne (2 ans)", className="card-text text-center text-muted")
            ])
        ], color="dark", outline=True), width=3),

        dbc.Col(dbc.Card([
            dbc.CardBody([
                html.H4(f"{df['ScoreCampagne_100'].mean():.1f}/100", className="card-title text-center", style={"color":"#0f3460","fontSize":"2em"}),
                html.P("Score moyen campagne", className="card-text text-center text-muted")
            ])
        ], color="dark", outline=True), width=3),
    ], className="mb-4"),

    # Tabs
    dbc.Tabs([

        # Tab 1 — Portefeuille
        dbc.Tab(label="🏠 Portefeuille", children=[
            dbc.Row([
                dbc.Col(dcc.Graph(id="hist_income"), width=6),
                dbc.Col(dcc.Graph(id="pie_education"), width=6),
            ], className="mt-3"),
            dbc.Row([
                dbc.Col(dcc.Graph(id="scatter_income_spend"), width=12)
            ])
        ]),

        # Tab 2 — Segmentation
        dbc.Tab(label="🧩 Segmentation", children=[
            dbc.Row([
                dbc.Col([
                    html.Label("Axe X :", className="text-light mt-3"),
                    dcc.Dropdown(id="seg_x",
                                 options=[{"label": c, "value": c} for c in
                                          ["Income","TotalSpend","Age","Recency","ScoreCampagne_100","RFM_score"]],
                                 value="Income", clearable=False,
                                 style={"color":"#000"}),
                    html.Label("Axe Y :", className="text-light mt-2"),
                    dcc.Dropdown(id="seg_y",
                                 options=[{"label": c, "value": c} for c in
                                          ["TotalSpend","Income","Age","Recency","ScoreCampagne_100","RFM_score"]],
                                 value="TotalSpend", clearable=False,
                                 style={"color":"#000"}),
                ], width=3),
                dbc.Col(dcc.Graph(id="seg_scatter"), width=9)
            ], className="mt-3"),
            dbc.Row([dbc.Col(dcc.Graph(id="seg_bar_kpi"), width=12)])
        ]),

        # Tab 3 — Campagnes
        dbc.Tab(label="📣 Campagnes", children=[
            dbc.Row([
                dbc.Col(dcc.Graph(id="cmp_rates"), width=6),
                dbc.Col(dcc.Graph(id="cmp_by_segment"), width=6),
            ], className="mt-3"),
            dbc.Row([dbc.Col(dcc.Graph(id="score_distribution"), width=12)])
        ]),

        # Tab 4 — KPI Exécutifs
        dbc.Tab(label="📋 KPI Exécutifs", children=[
            dbc.Row([
                dbc.Col([
                    html.Label("Filtrer par segment :", className="text-light mt-3"),
                    dcc.Dropdown(id="kpi_segment",
                                 options=[{"label": "Tous", "value": "Tous"}] +
                                         [{"label": s, "value": s} for s in SEGMENTS],
                                 value="Tous", clearable=False,
                                 style={"color":"#000"}),
                ], width=4)
            ]),
            dbc.Row([
                dbc.Col(dcc.Graph(id="kpi_rfm"), width=6),
                dbc.Col(dcc.Graph(id="kpi_channels"), width=6),
            ], className="mt-2"),
            dbc.Row([dbc.Col(html.Div(id="kpi_table"), width=12)])
        ]),

    ])

], fluid=True)


# ── Callbacks ─────────────────────────────────────────────────────────────────

@app.callback(
    Output("hist_income","figure"),
    Output("pie_education","figure"),
    Output("scatter_income_spend","figure"),
    Input("hist_income","id")
)
def tab_portfolio(_):
    fig1 = px.histogram(df, x="Income", color="SegmentLabel", nbins=50,
                        title="Distribution du Revenu par Segment",
                        color_discrete_sequence=PALETTE, template="plotly_dark")
    fig2 = px.pie(df, names="Education", title="Répartition par Niveau d'Éducation",
                  color_discrete_sequence=PALETTE, template="plotly_dark", hole=0.4)
    fig3 = px.scatter(df, x="Income", y="TotalSpend", color="SegmentLabel",
                      size="ScoreCampagne_100", size_max=15, opacity=0.6,
                      title="Income vs Dépenses — taille = Score Campagne",
                      color_discrete_sequence=PALETTE, template="plotly_dark",
                      hover_data=["Age","Education","RFM_score"])
    return fig1, fig2, fig3


@app.callback(
    Output("seg_scatter","figure"),
    Output("seg_bar_kpi","figure"),
    Input("seg_x","value"),
    Input("seg_y","value")
)
def tab_segmentation(x, y):
    fig1 = px.scatter(df, x=x, y=y, color="SegmentLabel",
                      title=f"Segmentation — {x} vs {y}",
                      color_discrete_sequence=PALETTE, template="plotly_dark",
                      opacity=0.5, hover_data=["Education","Age","Response"])

    seg_kpi = df.groupby("SegmentLabel").agg(
        Income=("Income","mean"),
        Depenses=("TotalSpend","mean"),
        Reponse=("Response",lambda x: x.mean()*100),
        RFM=("RFM_score","mean")
    ).reset_index()

    fig2 = go.Figure()
    for col, label in [("Income","Revenu moy"),("Depenses","Dépenses moy"),("Reponse","Taux réponse %"),("RFM","Score RFM")]:
        fig2.add_trace(go.Bar(name=label, x=seg_kpi["SegmentLabel"], y=seg_kpi[col]))
    fig2.update_layout(barmode="group", template="plotly_dark", title="KPI comparatif par Segment")
    return fig1, fig2


@app.callback(
    Output("cmp_rates","figure"),
    Output("cmp_by_segment","figure"),
    Output("score_distribution","figure"),
    Input("cmp_rates","id")
)
def tab_campaigns(_):
    cmp_cols = ["AcceptedCmp1","AcceptedCmp2","AcceptedCmp3","AcceptedCmp4","AcceptedCmp5","Response"]
    cmp_names = ["Cmp 1","Cmp 2","Cmp 3","Cmp 4","Cmp 5","Cmp 6 (cible)"]
    rates = [df[c].mean()*100 for c in cmp_cols]
    fig1 = px.bar(x=cmp_names, y=rates, color=cmp_names,
                  title="Taux d'Acceptation par Campagne (%)",
                  color_discrete_sequence=PALETTE*2, template="plotly_dark",
                  labels={"x":"Campagne","y":"Taux (%)"})

    seg_cmp = df.groupby("SegmentLabel")[cmp_cols].mean().reset_index()
    seg_cmp_melt = seg_cmp.melt(id_vars="SegmentLabel", value_vars=cmp_cols,
                                 var_name="Campagne", value_name="Taux")
    fig2 = px.bar(seg_cmp_melt, x="SegmentLabel", y="Taux", color="Campagne",
                  title="Taux d'Acceptation par Segment et Campagne",
                  barmode="group", template="plotly_dark")

    fig3 = px.histogram(df, x="ScoreCampagne_100", color="SegmentLabel",
                        nbins=40, title="Distribution du Score Campagne par Segment",
                        color_discrete_sequence=PALETTE, template="plotly_dark",
                        labels={"ScoreCampagne_100":"Score campagne (0-100)"})
    fig3.add_vline(x=40, line_dash="dash", line_color="white",
                   annotation_text="Seuil ciblage (40)", annotation_position="top right")
    return fig1, fig2, fig3


@app.callback(
    Output("kpi_rfm","figure"),
    Output("kpi_channels","figure"),
    Output("kpi_table","children"),
    Input("kpi_segment","value")
)
def tab_kpi(segment):
    dff = df if segment == "Tous" else df[df["SegmentLabel"] == segment]

    # RFM distribution
    fig1 = px.box(dff, x="SegmentLabel", y="RFM_score", color="SegmentLabel",
                  title="Distribution du Score RFM par Segment",
                  color_discrete_sequence=PALETTE, template="plotly_dark")

    # Canaux d'achat
    channels = {"Web":dff["NumWebPurchases"].mean(), "Catalogue":dff["NumCatalogPurchases"].mean(),
                "Magasin":dff["NumStorePurchases"].mean(), "Promos":dff["NumDealsPurchases"].mean()}
    fig2 = px.pie(values=list(channels.values()), names=list(channels.keys()),
                  title="Répartition des Achats par Canal",
                  color_discrete_sequence=PALETTE, template="plotly_dark", hole=0.35)

    # Table KPI
    kpi_df = pd.DataFrame({
        "KPI": ["Nb clients","Taux réponse (%)","Revenu moyen (€)","Dépense moy. (€)",
                "Score RFM moyen","Score campagne moyen","Recency moy. (j)"],
        "Valeur": [
            len(dff),
            f"{dff['Response'].mean()*100:.1f}%",
            f"{dff['Income'].mean():,.0f}",
            f"{dff['TotalSpend'].mean():,.0f}",
            f"{dff['RFM_score'].mean():.1f}/100",
            f"{dff['ScoreCampagne_100'].mean():.1f}/100",
            f"{dff['Recency'].mean():.0f}",
        ]
    })
    table = dash_table.DataTable(
        data=kpi_df.to_dict("records"),
        columns=[{"name": c, "id": c} for c in kpi_df.columns],
        style_table={"width":"50%","margin":"auto","marginTop":"20px"},
        style_header={"backgroundColor":"#0f3460","color":"white","fontWeight":"bold"},
        style_data={"backgroundColor":"#1a1a2e","color":"white"},
        style_cell={"textAlign":"center","padding":"10px"},
    )
    return fig1, fig2, table


if __name__ == "__main__":
    app.run(debug=True)
